In [1]:

import pandas as pd
import sqlite3
from pathlib import Path

# Find project folder
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Load your CSV files
sensor_df = pd.read_csv(
    project_root / "data/processed/engine_sensor_data.csv"
)

validation_df = pd.read_csv(
    project_root / "data/processed/validation_results.csv"
)

# Connect to SQLite database
conn = sqlite3.connect(
    project_root / "data/engine_analytics.db"
)

# Create SQL tables
sensor_df.to_sql(
    "engine_sensor_readings",
    conn,
    if_exists="replace",
    index=False
)

validation_df.to_sql(
    "validation_results",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully!")


Database created successfully!


In [2]:

query = """
SELECT
    engine_id,
    cycle,
    sensor_11
FROM engine_sensor_readings
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

print(result)


   engine_id  cycle  sensor_11
0          1      1      47.47
1          1      2      47.49
2          1      3      47.27
3          1      4      47.13
4          1      5      47.28
5          1      6      47.16
6          1      7      47.36
7          1      8      47.24
8          1      9      47.29
9          1     10      47.03


In [3]:

query = """
SELECT
    engine_id,
    cycle,
    sensor_11
FROM engine_sensor_readings
WHERE sensor_11 > 47.701
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

print(result)


   engine_id  cycle  sensor_11
0          1    134      47.75
1          1    137      47.89
2          1    147      47.80
3          1    149      47.83
4          1    152      47.72
5          1    153      47.74
6          1    154      47.73
7          1    157      47.72
8          1    159      47.76
9          1    160      47.84


In [4]:

query = """
SELECT
    engine_id,
    ROUND(AVG(sensor_11), 2) AS avg_sensor_11
FROM engine_sensor_readings
GROUP BY engine_id
ORDER BY engine_id
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

print(result)


   engine_id  avg_sensor_11
0          1          47.51
1          2          47.39
2          3          47.43
3          4          47.52
4          5          47.40
5          6          47.71
6          7          47.47
7          8          47.71
8          9          47.34
9         10          47.40


In [5]:

query = """
SELECT
    engine_id,
    MAX(cycle) AS total_cycles
FROM engine_sensor_readings
GROUP BY engine_id
ORDER BY total_cycles DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

print(result)


   engine_id  total_cycles
0         69           362
1         92           341
2         96           336
3         67           313
4         83           293
5          2           287
6         95           283
7         64           283
8         86           278
9         17           276


In [6]:

query = """
SELECT
    v.engine_id,
    MAX(s.cycle) AS total_cycles,
    v.detection_cycle,
    v.lead_time
FROM validation_results AS v
JOIN engine_sensor_readings AS s
    ON v.engine_id = s.engine_id
GROUP BY
    v.engine_id,
    v.detection_cycle,
    v.lead_time
ORDER BY v.engine_id
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

print(result)


   engine_id  total_cycles  detection_cycle  lead_time
0          7           259              216         43
1          9           201              192          9
2         12           170              129         41
3         13           163              132         31
4         14           180              124         56
5         15           207              184         23
6         20           234              169         65
7         23           168              137         31
8         36           158              127         31
9         37           170              148         22
